In [3]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

DB_USER = "postgres"
DB_PASSWORD = "Shanks@123"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "job_market_db"

connection_url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    result = connection.execute(text("SELECT version();"))
    print(result.fetchone()[0])

PostgreSQL 17.11 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit


In [4]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

connection_url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password="Shanks@123",
    host="localhost",
    port=5432,
    database="job_market_db"
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(result.fetchone()[0])

PostgreSQL 17.11 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit


In [5]:
import pandas as pd

df = pd.read_csv("../data/cleaned_jobs.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 97679
Columns: 22


In [8]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd

# PostgreSQL connection
connection_url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password="Shanks@123",
    host="localhost",
    port=5432,
    database="job_market_db"
)

engine = create_engine(connection_url)

# Load cleaned CSV
df = pd.read_csv("../data/cleaned_jobs.csv")

print("Rows to insert:", len(df))

Rows to insert: 97679


In [9]:
df.to_sql(
    "jobs",
    engine,
    if_exists="replace",
    index=False,
    chunksize=5000,
    method="multi"
)

print("✅ Jobs inserted into PostgreSQL")

✅ Jobs inserted into PostgreSQL


In [10]:
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM jobs")
    )
    
    print("Jobs in PostgreSQL:", result.scalar())

Jobs in PostgreSQL: 97679


In [11]:
pd.read_sql(
    """
    SELECT
        job_id,
        title,
        company_name,
        location,
        minimum_salary,
        maximum_salary
    FROM jobs
    LIMIT 10
    """,
    engine
)

,job_id,title,company_name,location,minimum_salary,maximum_salary
0,270925008041,Sr. HR Recruiter (NON IT),Orion,Kolkata(Chinar Park),200000.0,400000.0
1,270925007584,Fire And Safety Officer,"Apollo Hospitals International Limited, Ahmedabad","Gandhinagar, Ahmedabad",300000.0,500000.0
2,270925007492,Opening For Performance Marketing - Chennai,TVS Credit Services Ltd,Chennai,NaN,NaN
3,270925007443,Medical Billing Executive,GNR Global Services,"Mohali, Chandigarh, Kharar, Zirakpur",70000.0,200000.0
4,270925007430,Senior Group Product Manager - CNS Therapy,Cadila Pharmaceuticals,Ahmedabad,800000.0,1800000.0
5,270925007323,Senior Accountant,Vedaant Techno Art,"Noida, Greater Noida",350000.0,600000.0
6,270925007180,Maintenance Technician,Avalon Technologies,"Sriperumbudur, Chennai, Kanchipuram",NaN,NaN
7,270925006659,Officer/Engineer Technical,Balkrishna Industries (BKT),Aurangabad(Waluj),NaN,NaN
8,270925006525,Officer- Technical Lab,Balkrishna Industries (BKT),Aurangabad(Waluj),NaN,NaN
9,270925006315,Site Reliability Engineer,Coforge,"New Delhi, Gurugram, Greater Noida",NaN,NaN


In [12]:
query = """
SELECT
    COUNT(*) AS total_jobs,
    COUNT(minimum_salary) AS jobs_with_salary,
    COUNT(*) - COUNT(minimum_salary) AS jobs_without_salary
FROM jobs;
"""

pd.read_sql(query, engine)

,total_jobs,jobs_with_salary,jobs_without_salary
0,97679,33136,64543


In [13]:
query = """
SELECT
    currency,
    COUNT(*) AS job_count
FROM jobs
GROUP BY currency
ORDER BY job_count DESC;
"""

pd.read_sql(query, engine)

,currency,job_count
0,INR,97550
1,USD,129
